## Analys data


In [1]:
# pyspark init

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

26/03/05 23:46:39 WARN Utils: Your hostname, nouhayla-HP-EliteBook-835-G7-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.96.226 instead (on interface wlp1s0)
26/03/05 23:46:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/05 23:46:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/05 23:46:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/05 23:46:40 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
# loading data

df_bronze = (spark.read
             .format("csv")
             .option("header", "true")
             .load("../data/raw/data.csv"))

df_bronze.show()

+----------+---+------+------+---------------+-------------+------------------+--------------------+--------------------+-------------+------------------+------------------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|CustomerID|Age|Gender|Income|CampaignChannel| CampaignType|           AdSpend|    ClickThroughRate|      ConversionRate|WebsiteVisits|     PagesPerVisit|        TimeOnSite|SocialShares|EmailOpens|EmailClicks|PreviousPurchases|LoyaltyPoints|AdvertisingPlatform|AdvertisingTool|Conversion|
+----------+---+------+------+---------------+-------------+------------------+--------------------+--------------------+-------------+------------------+------------------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|      8000| 56|Female|136912|   Social Media|    Awareness| 6497.870068417766| 0.04391851073538301| 0.08803141207288108|            

In [3]:
# definir les chemins

input_path = "../data/raw/data.csv"
output_path = "../data/bronze/bronze_data"


df = spark.read.format("csv").option("header", "true").load(input_path)

df.write.mode("overwrite").parquet(output_path)

In [4]:
#  data types

df_bronze.printSchema()

root
 |-- CustomerID: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Income: string (nullable = true)
 |-- CampaignChannel: string (nullable = true)
 |-- CampaignType: string (nullable = true)
 |-- AdSpend: string (nullable = true)
 |-- ClickThroughRate: string (nullable = true)
 |-- ConversionRate: string (nullable = true)
 |-- WebsiteVisits: string (nullable = true)
 |-- PagesPerVisit: string (nullable = true)
 |-- TimeOnSite: string (nullable = true)
 |-- SocialShares: string (nullable = true)
 |-- EmailOpens: string (nullable = true)
 |-- EmailClicks: string (nullable = true)
 |-- PreviousPurchases: string (nullable = true)
 |-- LoyaltyPoints: string (nullable = true)
 |-- AdvertisingPlatform: string (nullable = true)
 |-- AdvertisingTool: string (nullable = true)
 |-- Conversion: string (nullable = true)



In [5]:
# data columns

df_bronze.columns

['CustomerID',
 'Age',
 'Gender',
 'Income',
 'CampaignChannel',
 'CampaignType',
 'AdSpend',
 'ClickThroughRate',
 'ConversionRate',
 'WebsiteVisits',
 'PagesPerVisit',
 'TimeOnSite',
 'SocialShares',
 'EmailOpens',
 'EmailClicks',
 'PreviousPurchases',
 'LoyaltyPoints',
 'AdvertisingPlatform',
 'AdvertisingTool',
 'Conversion']

In [6]:
df_bronze.count()

8000

In [7]:
# statistique descriptive

df_bronze.describe().show()

26/03/05 23:46:48 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------+------------------+---------------+------------+------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+----------------+------------------+-----------------+------------------+-------------------+---------------+-------------------+
|summary|        CustomerID|               Age|Gender|            Income|CampaignChannel|CampaignType|           AdSpend|    ClickThroughRate|      ConversionRate|    WebsiteVisits|     PagesPerVisit|        TimeOnSite|      SocialShares|      EmailOpens|       EmailClicks|PreviousPurchases|     LoyaltyPoints|AdvertisingPlatform|AdvertisingTool|         Conversion|
+-------+------------------+------------------+------+------------------+---------------+------------+------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+----------------+---------

In [8]:
# null values


from pyspark.sql.functions import col, sum as spark_sum, when
df_bronze.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
      for c in df_bronze.columns
      ]).show()

+----------+---+------+------+---------------+------------+-------+----------------+--------------+-------------+-------------+----------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|CustomerID|Age|Gender|Income|CampaignChannel|CampaignType|AdSpend|ClickThroughRate|ConversionRate|WebsiteVisits|PagesPerVisit|TimeOnSite|SocialShares|EmailOpens|EmailClicks|PreviousPurchases|LoyaltyPoints|AdvertisingPlatform|AdvertisingTool|Conversion|
+----------+---+------+------+---------------+------------+-------+----------------+--------------+-------------+-------------+----------+------------+----------+-----------+-----------------+-------------+-------------------+---------------+----------+
|         0|  0|     0|     0|              0|           0|      0|               0|             0|            0|            0|         0|           0|         0|          0|                0|            0|                  0|            

In [9]:
print("Nombres des lingne:", df_bronze.count())

Nombres des lingne: 8000


In [10]:
uniques_lignes = df_bronze.dropDuplicates().count()
print(f"Nombre de lignes uniques :{uniques_lignes}")

Nombre de lignes uniques :8000


In [11]:
df_silver = df_bronze.drop("ConversionRate")

In [12]:
df_silver = df_silver.write.mode("overwrite").parquet(
    "../data/silver/silver_data"
)

print("silver layer created successfully")

26/03/05 23:46:51 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


silver layer created successfully
